# Induction Evals — One-Hop (Range Query)
This notebook is the **one-hop** sibling of `induction_eval.ipynb`: instead of direct-succession pairs, each question asks whether a single sampled year could belong to a colour, so the intensional and extensional representations each require exactly **one hop** (one boundary check vs. one year-keyed lookup).

**Replication design:** each (archetype, info type) condition runs **R = 30 replicates** of the same one-hop True/False quiz, each under a fresh `ChromaticIntervalsConfig` seed (fresh interval history, colour labels, and False-year sampling, with that seed reused as the per-request decoding seed). The unit of observation is the **quiz**: its ~120 questions share one context and are dominated by a near-global True/False response bias, so they are not iid. A replicate re-runs the *whole* quiz under a new seed -- it never enlarges a single quiz. R = 30 mirrors `induction_eval.ipynb` / `notebooks/periodic`. Each replicate serializes to `results/one_hop_{archetype}_{info}/rep_{seed}.yaml` as soon as it is graded, so interrupted sections resume where they left off. All 30 replicates (seeds 1776–1805) are generated fresh under the current olmo/granite trio; the earlier single-seed run used a now-undeployable trio and was discarded, so there is no migrated `rep_1776.yaml` here.

In [ ]:
"""Generates the one-hop range quiz that all models are evaluated on."""

import string

import logging

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
load_dotenv("./keys.env", verbose=True)

from typing import Dict, Tuple

from smolbench.induction.chromatic import (
    ChromaticIntervalsConfig,
    Prompter,
    get_random_exclusive_quiz,
    one_hop_year_query_gen,
)

# Inference runs on a self-provisioned EC2 spot instance serving vLLM
# (smolbench/evals/ec2.py), exactly like notebooks/periodic. keys.env
# (gitignored) selects the provider and this experiment's static EC2 config:
# INFERENCE_PROVIDER=ec2, EC2_EXPERIMENT_TAG=chromatic-induction (isolates
# this experiment's instance from notebooks/periodic), and
# EC2_INSTANCE_TYPES=p5e.48xlarge,p5.48xlarge (8-GPU H200/H100 at tp=8: the
# 4x L40S g6e boxes the trio first ran on were decode-bandwidth bound; see
# the EC2_DEPLOY_SPECS comment in smolbench/evals/ec2.py). Provider dispatch
# and the EC2 state-file path are resolved at CALL time, so env only needs
# to be set before the first provision/evaluate call. The state-file path is
# set here: it must be anchored via __file__ (notebook kernels run with
# temp-dir cwds) and kept separate from periodic's .ec2_state.json.
import os
from pathlib import Path
import smolbench

os.environ["EC2_STATE_FILE"] = str(
    Path(smolbench.__file__).resolve().parents[1] / ".ec2_state_chromatic.json"
)

# Per-archetype model names: keys of EC2_DEPLOY_SPECS in
# smolbench/evals/ec2.py (each key is also vLLM's --served-model-name). An
# ungated ~32B English-centric trio -- no HF account/login/token needed.
DENSE_MODEL = "olmo-3.1-32b-instruct"  # allenai/Olmo-3.1-32B-Instruct (32B dense, non-reasoning)
COT_MODEL = "olmo-3.1-32b-think"       # allenai/Olmo-3.1-32B-Think (32B dense, always reasons)
MOE_MODEL = "granite-4.0-h-small"      # ibm-granite/granite-4.0-h-small (MoE 32B/~9B active, non-reasoning)
# Olmo-3.1-32B-Think's chat template force-opens <think>, so it reasons
# unconditionally; give the completion enough budget to finish thinking AND
# emit the answer. query() splits the think block into the reasoning channel
# client-side, so scoring sees only the answer.
COT_EXTRA_ARGS = {"max_completion_tokens": 8192}
# CoT runs must let the LONGEST chain finish on attempt 1: a tight client
# timeout + high concurrency censors long-CoT requests (they time out, retry,
# and survive only via the scheduler lottery -> the measured CoT-length
# distribution is cut off at the top, and the scored output stops being
# seed-deterministic). Low parallelism gives each chain enough GPU to finish
# well under a generous timeout, uniformly.
COT_MAX_PARALLEL = 16       # p5e/p5 (8-GPU H200/H100) has huge KV headroom + ~5-7x faster decode, so feed it wider
COT_REQUEST_TIMEOUT = 1200  # seconds: covers an 8192-token chain even at low decode tok/s

template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. The following lists the people who were $role and"
    " the years they were $role:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "During year $year, could $color have headed the $parade parade?"
)

extens_template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. The following lists each year and who was $role"
    " that year:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "During year $year, could $color have headed the $parade parade?"
)

# The one-hop query generator is the module builtin
# smolbench.induction.chromatic.one_hop_year_query_gen (imported above), for
# the same reason its succession sibling lives there: one importable, tested
# definition of the task.
QUERY_GEN = one_hop_year_query_gen

# --- Replication setup ---------------------------------------------------
# A replicate is the SAME one-hop True/False quiz regenerated
# under a fresh seed: fresh interval history + color labels (driven by
# ChromaticIntervalsConfig.seed) and fresh False-year sampling (query_gen
# reuses that seed), plus that same seed as the per-request decoding seed.
# The unit of observation is the quiz, not the question -- a quiz's ~120
# questions share one context and are dominated by a near-global True/False
# bias -- so power comes from replicating whole quizzes under new seeds,
# never from enlarging one quiz. R=30 mirrors notebooks/periodic and feeds
# power_analysis.py its between-quiz variance estimate (see that script).
R: int = 30
BASE_SEED: int = 1776  # base seed; all R replicates generated fresh (no migrated rep_1776)
REPLICATE_SEEDS: Tuple[int, ...] = tuple(BASE_SEED + r for r in range(R))
INFO_TYPES: Tuple[str, ...] = ("intens", "extens", "noise_intens")


def make_quizzes(seed: int) -> Dict[str, tuple]:
    """Generates one replicate's three info-type quizzes, keyed by info type.

    Chromatic quizzes are large -- every prompt embeds the full interval
    history (the extens listing alone is ~3000 lines) across ~120 questions
    -- so replicates are generated on demand per seed inside run_replicates,
    not all precomputed up front, to keep notebook memory bounded.
    """
    return dict(
        zip(
            INFO_TYPES,
            get_random_exclusive_quiz(
                ChromaticIntervalsConfig(
                    n=int(12 * 250),
                    intervals=250 // 4,
                    colors=45,
                    seed=seed,
                ),
                Prompter(
                    template,
                    {
                        "role": "Twislax",
                        "parade": "Gildane",
                    },
                    QUERY_GEN,
                    extens_template,
                ),
            ),
        )
    )


# First-replicate aliases for the Prompt Validation cells below (the only
# replicate generated eagerly; the rest are built per seed when run).
_base_quizzes: Dict[str, tuple] = make_quizzes(BASE_SEED)
intens_quiz, extens_quiz, noise_intens_quiz = (
    _base_quizzes["intens"],
    _base_quizzes["extens"],
    _base_quizzes["noise_intens"],
)


In [ ]:
# Replicated evaluation harness (shared: smolbench/evals/replicates.py).
# Each (archetype, info type, seed) replicate is serialized to
# results/{prefix}{tag}_{info}/rep_{seed}.yaml IMMEDIATELY after it is
# graded, so a spot interruption or kernel restart loses at most one
# replicate's work; reruns skip already-serialized replicates, making the
# archetype cells below idempotent and resumable. A seed's outstanding info
# types are pooled into ONE evaluate() call so the GPU stays saturated
# instead of draining to zero between per-info barriers.
#
# RESULTS_DIR is anchored via the installed package (like EC2_STATE_FILE in
# the first cell) rather than left cwd-relative: notebook kernels can run
# with a temp-dir cwd, and power_analysis.py reads this same results/ dir.
from smolbench.evals.replicates import ReplicateHarness

RESULTS_DIR: Path = (
    Path(smolbench.__file__).resolve().parents[1]
    / "notebooks"
    / "chromatic"
    / "results"
)

HARNESS = ReplicateHarness(
    results_dir=RESULTS_DIR,
    archetype_tags={DENSE_MODEL: "decode", COT_MODEL: "cot", MOE_MODEL: "moe"},
    make_quizzes=make_quizzes,
    seeds=REPLICATE_SEEDS,
    info_types=INFO_TYPES,
    # one_hop shares the results/ tree with induction_eval.ipynb; the
    # prefix namespaces its replicate dirs as results/one_hop_{tag}_{info}/
    # so the two experiments never collide (power_analysis.py reads only
    # the unprefixed dirs).
    prefix="one_hop_",
)
# The archetype section cells below call these directly.
run_replicates = HARNESS.run_replicates
summarize = HARNESS.summarize


In [ ]:
# Provision ONE EC2 spot instance for the whole experiment (the archetype
# sections below only swap which model its vLLM serves). This cell is
# idempotent: it reattaches via .ec2_state_chromatic.json / the
# smolbench:experiment tag instead of launching a second box, so it is safe
# after kernel restarts. NOTE: induction_eval.ipynb shares the same
# experiment tag, so both notebooks reuse one instance if run back-to-back.
#
# Safety nets if you forget the Teardown cell at the bottom: an on-instance
# watchdog terminates the box after EC2_IDLE_TIMEOUT_MIN (default 30) idle
# minutes, and an absolute EC2_MAX_LIFETIME_MIN (default 24h) backstop
# exists. That also means: provision right before running the sections, not
# hours ahead. Spot price is ~$30-45/h for p5e/p5 -- mind the meter.
from smolbench.evals.ec2 import (
    agent_status,
    provision_spot_instance,
    serve_model,
    shutdown_instance,
)

state = provision_spot_instance()
print(
    f"instance {state['instance_id']} ({state['instance_type']}) "
    f"in {state['availability_zone']} at {state['public_ip']}"
)


## Prompt Validation

In [ ]:
print(intens_quiz[0].prompt)

In [ ]:
print(extens_quiz[0].prompt)

In [ ]:
print(noise_intens_quiz[0].prompt)

## Decoder-Only Model
This section tests classical decoder-only models.

In [ ]:
# serve_model points the shared instance's vLLM at DENSE_MODEL (a container
# swap; the first serve waits on the checkpoint download, reruns hit the
# instance's cache). Runs all outstanding R replicates x 3 info types; safe
# to rerun after an interruption -- finished replicates are skipped. Exit
# leaves the instance running for the next section.
with serve_model(DENSE_MODEL):
    run_replicates(DENSE_MODEL)

In [ ]:
# Aggregate results over all serialized decode replicates.
summarize(DENSE_MODEL)

## CoT Model
This section tests a CoT model (allenai/Olmo-3.1-32B-Think — dense 32B; its chat template force-opens `<think>`, so it reasons unconditionally).

In [ ]:
# Swaps the instance's vLLM to COT_MODEL. Olmo-3.1-32B-Think reasons
# unconditionally (its chat template force-opens <think>); query() splits
# the think block into the reasoning channel client-side, so scoring sees
# only the answer. The wide max_parallel + long request_timeout are applied
# to every info type and replicate so the longest chain finishes on attempt
# 1 (a tight timeout censors long-CoT requests -> non-deterministic,
# top-truncated output).
with serve_model(COT_MODEL):
    run_replicates(
        COT_MODEL,
        extra_args=COT_EXTRA_ARGS,
        max_parallel=COT_MAX_PARALLEL,
        request_timeout=COT_REQUEST_TIMEOUT,
    )

In [ ]:
# Aggregate results over all serialized cot replicates.
summarize(COT_MODEL)

In [ ]:
# Reasoning-chain length analysis: word counts from the cached CoT YAMLs
# (~1.3 tokens/word for these tokenizers). A top-truncated distribution here
# flags a too-tight CoT timeout (see the request_timeout note above).
HARNESS.cot_chain_lengths()


## MoE Model
This section tests an MoE model.

In [ ]:
# Swaps the instance's vLLM to MOE_MODEL.
with serve_model(MOE_MODEL):
    run_replicates(MOE_MODEL)

In [ ]:
# Aggregate results over all serialized moe replicates.
summarize(MOE_MODEL)

# Teardown
Gracefully shuts the experiment's EC2 spot instance down. If this cell is forgotten, the instance still self-terminates: an on-instance watchdog fires after `EC2_IDLE_TIMEOUT_MIN` (default 30) minutes without inference traffic or control-agent activity, and an absolute `EC2_MAX_LIFETIME_MIN` (default 24h) `shutdown -h` backstop is scheduled at boot. Both run on the instance itself, so they work even if this notebook's kernel is gone.

In [ ]:
# Terminates the spot instance (and its EBS volume) and clears
# .ec2_state_chromatic.json. Also works after a kernel restart or a lost
# state file: it falls back to the smolbench:experiment instance tag.
shutdown_instance()